In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv
/kaggle/input/datasets/niraliivaghani/flipkart-product-customer-reviews-dataset/Dataset-SA.csv


In [2]:
import pandas as pd
df = pd.read_csv("/kaggle/input/datasets/niraliivaghani/flipkart-product-customer-reviews-dataset/Dataset-SA.csv")
df.head()

,product_name,product_price,Rate,Review,Summary,Sentiment
0,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,5,super!,great cooler excellent air flow and for this p...,positive
1,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,5,awesome,best budget 2 fit cooler nice cooling,positive
2,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,3,fair,the quality is good but the power of air is de...,positive
3,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,1,useless product,very bad product its a only a fan,negative
4,Candes 12 L Room/Personal Air Cooler??????(Whi...,3999,3,fair,ok ok product,neutral


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205052 entries, 0 to 205051
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   product_name   205052 non-null  object
 1   product_price  205052 non-null  object
 2   Rate           205052 non-null  object
 3   Review         180388 non-null  object
 4   Summary        205041 non-null  object
 5   Sentiment      205052 non-null  object
dtypes: object(6)
memory usage: 9.4+ MB


In [8]:
df.shape

(205052, 6)

In [9]:
df['Sentiment'].value_counts().reset_index()

,Sentiment,count
0,positive,166581
1,negative,28232
2,neutral,10239


In [3]:
X = df[["Summary", "Sentiment"]]
X.head()

,Summary,Sentiment
0,great cooler excellent air flow and for this p...,positive
1,best budget 2 fit cooler nice cooling,positive
2,the quality is good but the power of air is de...,positive
3,very bad product its a only a fan,negative
4,ok ok product,neutral


In [18]:
raw_sentence = X.loc[0, "Summary"]
print(raw_sentence)

great cooler excellent air flow and for this price its so amazing and unbelievablejust love it


In [19]:
import re
sentence = raw_sentence.lower()
cleaned_sentence = re.sub(r'[^a-zA-Z]', ' ', sentence)
print("clean Sentence: \n", cleaned_sentence)

clean Sentence: 
 great cooler excellent air flow and for this price its so amazing and unbelievablejust love it


In [14]:
cleaned_sentence.split()

['great',
 'cooler',
 'excellent',
 'air',
 'flow',
 'and',
 'for',
 'this',
 'price',
 'its',
 'so',
 'amazing',
 'and',
 'unbelievablejust',
 'love',
 'it']

In [15]:
print(len(cleaned_sentence.split()))

16


In [17]:
import nltk
from nltk.corpus import stopwords
nltk.download("stopwords")
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [23]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
sentence = cleaned_sentence.split()
processed_word = []
for word in sentence:
    if word not in stop_words:
        lemmatized_word = lemmatizer.lemmatize(word)
        processed_word.append(lemmatized_word)
        result = " ".join(processed_word)
print(f"raw sentence: {raw_sentence}")
print(f"\npreprocessed sentence: {result}")

raw sentence: great cooler excellent air flow and for this price its so amazing and unbelievablejust love it

preprocessed sentence: great cooler excellent air flow price amazing unbelievablejust love


In [24]:
from gensim.models import Word2Vec
word2vec_model = Word2Vec(
    sentences=result,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

def document_vector(tokens):

    vectors = []

    for word in tokens:

        if word in word2vec_model.wv:

            vectors.append(word2vec_model.wv[word])

    if len(vectors) == 0:

        return np.zeros(100)

    return np.mean(vectors, axis=0)

In [31]:
import numpy as np
x = np.array(document_vector(result))

In [32]:
print(x)

[-1.9672769e-03  1.8561544e-03  8.7967294e-04  2.0538927e-03
  2.4210426e-04 -1.5546768e-03  1.7768701e-03  3.7822910e-03
 -3.0366157e-03 -2.2818602e-03  1.2174733e-03 -1.8661298e-03
 -5.7993311e-04  1.1163284e-03  9.1932563e-04 -6.6793669e-04
  2.8859971e-03  1.5325252e-03 -2.9500846e-03 -4.2359871e-03
  5.7272962e-04 -3.1177021e-04  4.1180444e-03 -9.5470308e-04
  9.9955557e-04  2.5789134e-04 -7.8894437e-04  1.9297886e-03
 -2.0823423e-03  1.1790198e-03  1.9690341e-03 -1.9267688e-03
  7.8715332e-04 -2.6172667e-03 -8.3761750e-04  1.1358421e-03
  1.9375103e-03  2.6650817e-04 -4.3162469e-05 -1.5550927e-04
  7.0327945e-04 -6.3205470e-04 -2.7032669e-03  8.6709397e-04
  8.7022415e-04  5.2267069e-04 -1.0113554e-03  2.0292576e-04
  6.0668238e-04  1.2428600e-03  3.9422524e-04 -1.6999925e-03
 -1.3694630e-03 -8.0344436e-04 -1.3560118e-03 -9.6488942e-04
  1.3276838e-03 -1.3165304e-03 -8.5012813e-04  7.3766703e-04
 -5.7000254e-04 -2.2327357e-04  2.5061856e-03 -1.2499604e-03
 -1.2974318e-03  2.86310

In [5]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X["encode_sentiment"] = le.fit_transform(X["Sentiment"])
X.head()

/tmp/ipykernel_58/2215007076.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["encode_sentiment"] = le.fit_transform(X["Sentiment"])


,Summary,Sentiment,encode_sentiment
0,great cooler excellent air flow and for this p...,positive,2
1,best budget 2 fit cooler nice cooling,positive,2
2,the quality is good but the power of air is de...,positive,2
3,very bad product its a only a fan,negative,0
4,ok ok product,neutral,1
